In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path().resolve().parent))
import pandas as pd

### Connection to the DB 

In [2]:
from src.utils.db import connect_to_db

engine=connect_to_db()

parameters of DATABASE sucessfuly loaded 
connection sucessful


### Loading F5_2_energy_emissions table

In [23]:
query_f5_2= """SELECT * FROM bronze.f5_2_lcp_energy_emissions 
WHERE "countryName"='Belgium' """

In [24]:
f5_2be=pd.read_sql(query_f5_2, engine)

In [5]:
f5_2be.info()

<class 'pandas.DataFrame'>
RangeIndex: 10521 entries, 0 to 10520
Data columns (total 14 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   PublicationDate                            10521 non-null  str    
 1   countryName                                10521 non-null  str    
 2   reportingYear                              10521 non-null  int64  
 3   LCPInspireId                               10521 non-null  str    
 4   installationPartName                       10521 non-null  str    
 5   installationPartNameConfidentialityReason  0 non-null      object 
 6   City_Of_Facility                           10456 non-null  str    
 7   addressConfidentialityReason               0 non-null      object 
 8   Longitude                                  10521 non-null  float64
 9   Latitude                                   10521 non-null  float64
 10  featureType                      

In [6]:
f5_2be.columns

Index(['PublicationDate', 'countryName', 'reportingYear', 'LCPInspireId',
       'installationPartName', 'installationPartNameConfidentialityReason',
       'City_Of_Facility', 'addressConfidentialityReason', 'Longitude',
       'Latitude', 'featureType', 'unit', 'featureValue',
       'confidentialityReason'],
      dtype='str')

In [34]:
col_to_keep= ['reportingYear', 'LCPInspireId',
       'Longitude',
       'Latitude', 'featureType', 'featureValue']

In [35]:
f5_2be=f5_2be[col_to_keep].copy()

In [38]:
f5_2be[f5_2be['featureType']=='NaturalGas'].head()

,reportingYear,LCPInspireId,Longitude,Latitude,featureType,featureValue
2720,2016,https://data.ied_registry.omgeving.vlaanderen....,4.321080,51.26737,NaturalGas,0.000000
2773,2016,BE.EEA/BE0023.PART,5.492726,50.93684,NaturalGas,50.760000
2805,2019,https://data.ied_registry.omgeving.vlaanderen....,4.326100,51.27146,NaturalGas,10824.603293
2820,2021,https://data.ied_registry.omgeving.vlaanderen....,4.296820,51.31586,NaturalGas,1363.747800
2843,2017,BE.WA/115010101.PART,5.912952,50.25002,NaturalGas,0.000000


In [17]:
f5_2be['featureType'].value_counts()

featureType
LCPCharacteristics    1632
NOX                    816
DUST                   816
SO2                    816
NaturalGas             806
LiquidFuels            805
OtherGases             805
Biomass                805
Lignite                805
Coal                   805
OtherSolidFuels        805
Peat                   805
Name: count, dtype: int64

In [18]:
f5_2be.head()

,reportingYear,LCPInspireId,installationPartName,City_Of_Facility,Longitude,Latitude,featureType,unit,featureValue
0,2018,https://data.ied_registry.omgeving.vlaanderen....,VPK PAPER,Dendermonde,4.068840,51.015210,LCPCharacteristics,MW,65.0
1,2016,https://data.ied_registry.omgeving.vlaanderen....,TOTALENERGIES REFINERY ANTWERP_LCP 2,Antwerpen,4.326460,51.266090,LCPCharacteristics,MW,259.0
2,2016,https://data.ied_registry.omgeving.vlaanderen....,TURBO-JET ZEEBRUGGE,Brugge,3.192240,51.318090,LCPCharacteristics,MW,80.0
3,2021,https://data.ied_registry.omgeving.vlaanderen....,TEREOS STARCH_SWEETENERS BELGIUM_LCP 1,Aalst,4.043870,50.937800,LCPCharacteristics,MW,163.0
4,2020,BE.WA/095010101.PART,Installations de combustion,Wanze,5.207238,50.528587,LCPCharacteristics,MW,125.0


In [19]:
#filtrage par combustible
combustible=['NaturalGas', 'LiquidFuels', 'OtherGases', 'Biomass', 'Lignite', 'Coal', 'OtherSolidFuels', 'Peat']
mask=f5_2be['featureType'].isin(combustible)
f5_2_be_filtered=f5_2be[mask]

In [20]:
f5_2_be_filtered.head()

,reportingYear,LCPInspireId,installationPartName,City_Of_Facility,Longitude,Latitude,featureType,unit,featureValue
2723,2016,https://data.ied_registry.omgeving.vlaanderen....,TOTALENERGIES REFINERY ANTWERP_LCP 6,Antwerpen,4.321080,51.267370,NaturalGas,TJ,0.0
2724,2020,https://data.ied_registry.omgeving.vlaanderen....,ZANDVLIET POWER - TERREIN BASF,Antwerpen,4.266880,51.368860,LiquidFuels,TJ,0.0
2750,2019,BE.WA/115010101.PART,Cierreux I (Turbo jet back-up),Bovigny,5.912952,50.250020,OtherGases,TJ,0.0
2761,2016,BE.WA/049010702.PART,Solvay GT1A,Jemeppe-Sur-Sambre,4.662693,50.447224,Biomass,TJ,0.0
2762,2017,https://data.ied_registry.omgeving.vlaanderen....,LUMINUS Ham_LCP 2,Gent,3.736470,51.059740,Lignite,TJ,0.0


In [37]:
f5_2_be_filtered['unit'].value_counts()

unit
TJ    6441
Name: count, dtype: int64

In [21]:
f5_2_be_filtered.columns

Index(['reportingYear', 'LCPInspireId', 'installationPartName',
       'City_Of_Facility', 'Longitude', 'Latitude', 'featureType', 'unit',
       'featureValue'],
      dtype='str')

In [22]:
#pivot
f5_2_be_pivoted= f5_2_be_filtered.pivot_table(index=['reportingYear','LCPInspireId','installationPartName',
       'City_Of_Facility', 'Longitude', 'Latitude'],columns='featureType',values='featureValue')

In [23]:
f5_2_be_pivoted.head()

featureType                                                                                                  Biomass  \
reportingYear LCPInspireId          installationPartName               City_Of_Facility Longitude Latitude             
2016          BE.BRU/100010002.PART TURBOJET BUDA                      Bruxelles        4.411200  50.906700     0.00   
              BE.BRU/100010003.PART TURBOJET VOLTA                     Bruxelles        4.395500  50.813900     0.00   
              BE.EEA/BE0015.PART    Centrale Elec. Electrabel Awirs t5 FLEMALLE-HAUTE   5.422640  50.586396     0.00   
              BE.EEA/BE0022.PART    LANGERLO_LCP1                      Genk             5.492726  50.936840    86.26   
              BE.EEA/BE0023.PART    LANGERLO _ LCP2                    Genk             5.492726  50.936840   115.42   

featureType                                                                                                     Coal  \
reportingYear LCPInspireId          installationPartName               City_Of_Facility Longitude Latitude             
2016          BE.BRU/100010002.PART TURBOJET BUDA                      Bruxelles        4.411200  50.906700     0.00   
              BE.BRU/100010003.PART TURBOJET VOLTA                     Bruxelles        4.395500  50.813900     0.00   
              BE.EEA/BE0015.PART    Centrale Elec. Electrabel Awirs t5 FLEMALLE-HAUTE   5.422640  50.586396     0.00   
              BE.EEA/BE0022.PART    LANGERLO_LCP1                      Genk             5.492726  50.936840  1359.57   
              BE.EEA/BE0023.PART    LANGERLO _ LCP2                    Genk             5.492726  50.936840  1720.94   

featureType                                                                                                  Lignite  \
reportingYear LCPInspireId          installationPartName               City_Of_Facility Longitude Latitude             
2016          BE.BRU/100010002.PART TURBOJET BUDA                      Bruxelles        4.411200  50.906700      0.0   
              BE.BRU/100010003.PART TURBOJET VOLTA                     Bruxelles        4.395500  50.813900      0.0   
              BE.EEA/BE0015.PART    Centrale Elec. Electrabel Awirs t5 FLEMALLE-HAUTE   5.422640  50.586396      0.0   
              BE.EEA/BE0022.PART    LANGERLO_LCP1                      Genk             5.492726  50.936840      0.0   
              BE.EEA/BE0023.PART    LANGERLO _ LCP2                    Genk             5.492726  50.936840      0.0   

featureType                                                                                                  LiquidFuels  \
reportingYear LCPInspireId          installationPartName               City_Of_Facility Longitude Latitude                 
2016          BE.BRU/100010002.PART TURBOJET BUDA                      Bruxelles        4.411200  50.906700         4.15   
              BE.BRU/100010003.PART TURBOJET VOLTA                     Bruxelles        4.395500  50.813900         2.19   
              BE.EEA/BE0015.PART    Centrale Elec. Electrabel Awirs t5 FLEMALLE-HAUTE   5.422640  50.586396         0.00   
              BE.EEA/BE0022.PART    LANGERLO_LCP1                      Genk             5.492726  50.936840         0.00   
              BE.EEA/BE0023.PART    LANGERLO _ LCP2                    Genk             5.492726  50.936840         0.00   

featureType                                                                                                  NaturalGas  \
reportingYear LCPInspireId          installationPartName               City_Of_Facility Longitude Latitude                
2016          BE.BRU/100010002.PART TURBOJET BUDA                      Bruxelles        4.411200  50.906700        0.00   
              BE.BRU/100010003.PART TURBOJET VOLTA                     Bruxelles        4.395500  50.813900        0.00   
              BE.EEA/BE0015.PART    Centrale Elec. Electrabel Awirs t5 FLEMALLE-HAUTE   5.422640  50.586396        0

In [24]:
f5_2_be_pivoted=f5_2_be_pivoted.reset_index()

In [25]:
f5_2_be_pivoted.head()

featureType,reportingYear,LCPInspireId,installationPartName,City_Of_Facility,Longitude,Latitude,Biomass,Coal,Lignite,LiquidFuels,NaturalGas,OtherGases,OtherSolidFuels,Peat
0,2016,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.411200,50.906700,0.00,0.00,0.0,4.15,0.00,0.0,0.0,0.0
1,2016,BE.BRU/100010003.PART,TURBOJET VOLTA,Bruxelles,4.395500,50.813900,0.00,0.00,0.0,2.19,0.00,0.0,0.0,0.0
2,2016,BE.EEA/BE0015.PART,Centrale Elec. Electrabel Awirs t5,FLEMALLE-HAUTE,5.422640,50.586396,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0
3,2016,BE.EEA/BE0022.PART,LANGERLO_LCP1,Genk,5.492726,50.936840,86.26,1359.57,0.0,0.00,42.28,0.0,0.0,0.0
4,2016,BE.EEA/BE0023.PART,LANGERLO _ LCP2,Genk,5.492726,50.936840,115.42,1720.94,0.0,0.00,50.76,0.0,0.0,0.0


In [26]:
f5_2_be_pivoted.columns.name = None

In [27]:
f5_2_be_pivoted.head()

,reportingYear,LCPInspireId,installationPartName,City_Of_Facility,Longitude,Latitude,Biomass,Coal,Lignite,LiquidFuels,NaturalGas,OtherGases,OtherSolidFuels,Peat
0,2016,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.411200,50.906700,0.00,0.00,0.0,4.15,0.00,0.0,0.0,0.0
1,2016,BE.BRU/100010003.PART,TURBOJET VOLTA,Bruxelles,4.395500,50.813900,0.00,0.00,0.0,2.19,0.00,0.0,0.0,0.0
2,2016,BE.EEA/BE0015.PART,Centrale Elec. Electrabel Awirs t5,FLEMALLE-HAUTE,5.422640,50.586396,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0
3,2016,BE.EEA/BE0022.PART,LANGERLO_LCP1,Genk,5.492726,50.936840,86.26,1359.57,0.0,0.00,42.28,0.0,0.0,0.0
4,2016,BE.EEA/BE0023.PART,LANGERLO _ LCP2,Genk,5.492726,50.936840,115.42,1720.94,0.0,0.00,50.76,0.0,0.0,0.0


In [28]:
f5_2_be_pivoted= f5_2_be_pivoted.rename(str.lower, axis='columns')

In [29]:
f5_2_be_pivoted.head()

,reportingyear,lcpinspireid,installationpartname,city_of_facility,longitude,latitude,biomass,coal,lignite,liquidfuels,naturalgas,othergases,othersolidfuels,peat
0,2016,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.411200,50.906700,0.00,0.00,0.0,4.15,0.00,0.0,0.0,0.0
1,2016,BE.BRU/100010003.PART,TURBOJET VOLTA,Bruxelles,4.395500,50.813900,0.00,0.00,0.0,2.19,0.00,0.0,0.0,0.0
2,2016,BE.EEA/BE0015.PART,Centrale Elec. Electrabel Awirs t5,FLEMALLE-HAUTE,5.422640,50.586396,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0
3,2016,BE.EEA/BE0022.PART,LANGERLO_LCP1,Genk,5.492726,50.936840,86.26,1359.57,0.0,0.00,42.28,0.0,0.0,0.0
4,2016,BE.EEA/BE0023.PART,LANGERLO _ LCP2,Genk,5.492726,50.936840,115.42,1720.94,0.0,0.00,50.76,0.0,0.0,0.0


In [30]:

# Exemple de normalisation minimale des noms
def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).lower().strip()
    for ch in ["-", "/", "(", ")", ",", ".", ";", ":", "'"]:
        x = x.replace(ch, " ")
    x = " ".join(x.split())
    return x

f5_2_be_pivoted[['installationpartname','city_of_facility']]= f5_2_be_pivoted[['installationpartname','city_of_facility']].map(normalize_text)


In [31]:
f5_2_be_pivoted.head()

,reportingyear,lcpinspireid,installationpartname,city_of_facility,longitude,latitude,biomass,coal,lignite,liquidfuels,naturalgas,othergases,othersolidfuels,peat
0,2016,BE.BRU/100010002.PART,turbojet buda,bruxelles,4.411200,50.906700,0.00,0.00,0.0,4.15,0.00,0.0,0.0,0.0
1,2016,BE.BRU/100010003.PART,turbojet volta,bruxelles,4.395500,50.813900,0.00,0.00,0.0,2.19,0.00,0.0,0.0,0.0
2,2016,BE.EEA/BE0015.PART,centrale elec electrabel awirs t5,flemalle haute,5.422640,50.586396,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0
3,2016,BE.EEA/BE0022.PART,langerlo_lcp1,genk,5.492726,50.936840,86.26,1359.57,0.0,0.00,42.28,0.0,0.0,0.0
4,2016,BE.EEA/BE0023.PART,langerlo _ lcp2,genk,5.492726,50.936840,115.42,1720.94,0.0,0.00,50.76,0.0,0.0,0.0


In [33]:
print(f5_2_be_pivoted['biomass'].max())
print(f5_2_be_pivoted['coal'].max())
print(f5_2_be_pivoted['liquidfuels'].max())
print(f5_2_be_pivoted['naturalgas'].max())
print(f5_2_be_pivoted['othergases'].max())
print(f5_2_be_pivoted['othersolidfuels'].max())
print(f5_2_be_pivoted['peat'].max())

15328.45
1720.94
836.908
34508.988903
19798.88952
10068.48128
0.0


In [34]:
f5_2_be_pivoted.columns

Index(['reportingyear', 'lcpinspireid', 'installationpartname',
       'city_of_facility', 'longitude', 'latitude', 'biomass', 'coal',
       'lignite', 'liquidfuels', 'naturalgas', 'othergases', 'othersolidfuels',
       'peat'],
      dtype='str')

In [33]:
f5_2_be_pivoted.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   reportingyear         801 non-null    int64  
 1   lcpinspireid          801 non-null    str    
 2   installationpartname  801 non-null    str    
 3   city_of_facility      801 non-null    str    
 4   longitude             801 non-null    float64
 5   latitude              801 non-null    float64
 6   biomass               800 non-null    float64
 7   coal                  800 non-null    float64
 8   lignite               800 non-null    float64
 9   liquidfuels           800 non-null    float64
 10  naturalgas            801 non-null    float64
 11  othergases            800 non-null    float64
 12  othersolidfuels       800 non-null    float64
 13  peat                  800 non-null    float64
dtypes: float64(10), int64(1), str(3)
memory usage: 87.7 KB


In [ ]:
import sys
sys.path.append('/home/olivierpi/technofutur/15-final_project/belgian-environement-monitor')
from src.silver import transform_data_f5_2

col=['reportingYear','LCPInspireId','installationPartName',
       'City_Of_Facility', 'Longitude', 'Latitude']
f5_2_fonction=transform_data_f5_2(f5_2be,col_to_keep,'featureType',col_combustible=combustible,index=col,col_pivot='featureType',value_pivot='featureValue',normalize_col=['installationpartname','city_of_facility'] )


ModuleNotFoundError: No module named 'utils'